In [1]:
# 1: Setup

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchvision.models as models
from tqdm.auto import tqdm

import numpy as np
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
Torchvision: 0.29.1+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
!pip install -q onnx onnxscript onnxruntime

In [4]:
# 2: Dataset

from torchvision import transforms

# Training
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

# Validation
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

# Test
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

In [5]:
import torch
import torchvision
from torch.utils.data import DataLoader, random_split

full_train_dataset = torchvision.datasets.CIFAR100(
    root='./data',
    train=True,
    transform=transform_train,
    download=True
)

full_val_dataset = torchvision.datasets.CIFAR100(
    root='./data',
    train=True,
    transform=transform_val,
    download=True
)

# Split training data to 45.000 / 5.000
train_size = 45000
val_size = 5000

train_indices, val_indices = random_split(
    range(len(full_train_dataset)),
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Create Subsets
train_dataset = torch.utils.data.Subset(
    full_train_dataset, train_indices.indices
)

val_dataset = torch.utils.data.Subset(
    full_val_dataset, val_indices.indices
)

# Test dataset
test_dataset = torchvision.datasets.CIFAR100(
    root='./data',
    train=False,
    transform=transform_test,
    download=True
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

# Control

images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("Classes:", len(full_train_dataset.classes))

100%|██████████| 169M/169M [18:32<00:00, 152kB/s]


Images: torch.Size([64, 3, 224, 224])
Labels: torch.Size([64])
Train samples: 45000
Validation samples: 5000
Test samples: 10000
Classes: 100


In [6]:
# 3: Model

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 114MB/s]


In [ ]:
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 100)
model = model.to(device)

In [12]:
# 4: Training

NUM_EPOCHS = 20

loss = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params=model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6)

In [ ]:
import time

best_acc = 0.0

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    # Training
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")

    for images, labels in train_bar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        batch_loss = loss(outputs, labels)

        batch_loss.backward()
        optimizer.step()

        running_loss += batch_loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

        train_bar.set_postfix(loss=f"{batch_loss.item():.4f}")

    train_loss = running_loss / train_total
    train_acc = 100 * train_correct / train_total

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]")

    with torch.inference_mode():
        for images, labels in val_bar:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            outputs = model(images)
            batch_loss = loss(outputs, labels)

            val_loss += batch_loss.item() * images.size(0)
            predicted = outputs.argmax(dim=1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            val_bar.set_postfix(loss=f"{batch_loss.item():.4f}")

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    # Scheduler
    scheduler.step()

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_resnet18_cifar100.pth")
        saved_msg = "--> [Best Model Saved!]"
    else:
        saved_msg = ""

    epoch_time = time.time() - start_time

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS:02d}] ({epoch_time:.1f}s) | "
        f"Train Loss: {train_loss:.4f} | Train Acc: %{train_acc:.2f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: %{val_acc:.2f} {saved_msg}"
    )

print(f"\nTraining completed! Highest Validation Accuracy: %{best_acc:.2f}")


Epoch 1/20 [Train]:   0%|          | 0/704 [00:00<?, ?it/s]

Epoch 1/20 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch [01/20] (150.5s) | Train Loss: 1.8625 | Train Acc: %55.58 | Val Loss: 0.9859 | Val Acc: %71.80 --> [Best Model Saved!]


Epoch 2/20 [Train]:   0%|          | 0/704 [00:00<?, ?it/s]

Epoch 2/20 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch [02/20] (156.5s) | Train Loss: 0.8847 | Train Acc: %75.47 | Val Loss: 0.8157 | Val Acc: %76.16 --> [Best Model Saved!]


... (50 outputs left out)

Epoch [19/20] (155.2s) | Train Loss: 0.0136 | Train Acc: %99.87 | Val Loss: 0.7008 | Val Acc: %83.40 --> [Best Model Saved!]


Epoch 20/20 [Train]:   0%|          | 0/704 [00:00<?, ?it/s]

Epoch 20/20 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch [20/20] (155.9s) | Train Loss: 0.0131 | Train Acc: %99.90 | Val Loss: 0.7000 | Val Acc: %83.44 --> [Best Model Saved!]

Eğitim tamamlandı! En yüksek Validation Accuracy: %83.44


In [ ]:
checkpoint = torch.load(
    "best_resnet18_cifar100.pth",
    map_location="cpu",
    weights_only=True
)

if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict)

model = model.to("cpu")
model.eval()

print("Best checkpoint issuccessfully loaded!")

Best checkpoint başarıyla yüklendi!


In [15]:
# 5: ONNX Export
model = model.to("cpu")
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [ ]:
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    "resnet18_finetuned_fp32.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    },
    opset_version=17
)

print("Fine-tuned ONNX FP32 export is completed!")

/tmp/ipykernel_1664/2710223498.py:3: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1116 20:00:40.740000 1664 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:55: 

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Fine-tuned ONNX FP32 export tamamlandı!


In [ ]:
import onnx

onnx_model = onnx.load("resnet18_finetuned_fp32.onnx")
onnx.checker.check_model(onnx_model)

print("ONNX model is valid!")

ONNX model başarıyla doğrulandı!


In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(
    "resnet18_finetuned_fp32.onnx",
    providers=["CPUExecutionProvider"]
)

print("ONNX Runtime session is successfully created!")

ONNX Runtime session başarıyla oluşturuldu!


In [19]:
import numpy as np

test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

with torch.inference_mode():
    pytorch_output = model(
        torch.from_numpy(test_input)
    ).numpy()

onnx_output = session.run(
    ["output"],
    {"input": test_input}
)[0]

max_diff = np.max(np.abs(pytorch_output - onnx_output))
mean_diff = np.mean(np.abs(pytorch_output - onnx_output))

print(f"Maximum Difference: {max_diff:.8f}")
print(f"Mean Difference: {mean_diff:.8f}")

pytorch_prediction = pytorch_output.argmax(axis=1)
onnx_prediction = onnx_output.argmax(axis=1)

print(
    "Prediction Agreement:",
    np.mean(pytorch_prediction == onnx_prediction) * 100,
    "%"
)

Maximum Difference: 0.00000358
Mean Difference: 0.00000100
Prediction Agreement: 100.0 %


In [20]:
import numpy as np
from onnxruntime.quantization import CalibrationDataReader

In [21]:
class CIFAR100CalibrationReader(CalibrationDataReader):

    def __init__(self, data_loader, input_name, max_batches=16):
        self.data_loader = data_loader
        self.input_name = input_name
        self.max_batches = max_batches
        self.iterator = iter(data_loader)
        self.batch_count = 0

    def get_next(self):

        if self.batch_count >= self.max_batches:
            return None

        try:
            images, _ = next(self.iterator)
        except StopIteration:
            return None

        self.batch_count += 1

        return {
            self.input_name: images.numpy().astype(np.float32)
        }

    def rewind(self):
        self.iterator = iter(self.data_loader)
        self.batch_count = 0

In [22]:
input_name = session.get_inputs()[0].name

calibration_reader = CIFAR100CalibrationReader(
    data_loader=val_loader,
    input_name=input_name,
    max_batches=16
)

calibration_reader.rewind()

In [23]:
sample = calibration_reader.get_next()

print("Input shape:", sample[input_name].shape)
print("Input dtype:", sample[input_name].dtype)

Input shape: (64, 3, 224, 224)
Input dtype: float32


In [24]:
from onnxruntime.quantization import (
    quantize_static,
    QuantFormat,
    QuantType,
    CalibrationMethod
)

In [ ]:
quantize_static(
    model_input="resnet18_finetuned_fp32.onnx",
    model_output="resnet18_finetuned_int8.onnx",
    calibration_data_reader=calibration_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    calibrate_method=CalibrationMethod.MinMax,
    per_channel=True
)

print("Fine-tuned INT8 Quantization is completed!")

Fine-tuned INT8 Quantization tamamlandı!


In [ ]:
import onnxruntime as ort

# ONNX FP32
fp32_session = ort.InferenceSession(
    "resnet18_finetuned_fp32.onnx",
    providers=["CPUExecutionProvider"]
)

# ONNX INT8
int8_session = ort.InferenceSession(
    "resnet18_finetuned_int8.onnx",
    providers=["CPUExecutionProvider"]
)

print("PyTorch FP32 model is ready!")
print("ONNX FP32 session is ready!")
print("ONNX INT8 session is ready!")

PyTorch FP32 model hazır!
ONNX FP32 session hazır!
ONNX INT8 session hazır!


In [27]:
def pytorch_inference(model, images):

    model.eval()

    with torch.inference_mode():
        outputs = model(images)
        predictions = outputs.argmax(dim=1)

    return predictions

In [28]:
def onnx_inference(session, images):

    outputs = session.run(
        ["output"],
        {"input": images.numpy().astype(np.float32)}
    )[0]

    predictions = outputs.argmax(axis=1)

    return predictions

In [29]:
def onnx_int8_inference(session, images):

    outputs = session.run(
        ["output"],
        {"input": images.numpy().astype(np.float32)}
    )[0]

    predictions = outputs.argmax(axis=1)

    return predictions

In [30]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

def evaluate_model(inference_fn, model, data_loader, model_name="Model"):

    all_preds = []
    all_labels = []

    for images, labels in tqdm(data_loader, desc=f"Evaluating {model_name}"):

        images = images.cpu()

        predictions = inference_fn(model, images)

        all_preds.extend(predictions.tolist())
        all_labels.extend(labels.tolist())

    accuracy = accuracy_score(all_labels, all_preds)

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return accuracy, macro_f1

In [31]:
pytorch_acc, pytorch_f1 = evaluate_model(
    pytorch_inference, model, val_loader, "PyTorch FP32"
)

onnx_acc, onnx_f1 = evaluate_model(
    onnx_inference, fp32_session, val_loader, "ONNX FP32"
)

int8_acc, int8_f1 = evaluate_model(
    onnx_int8_inference, int8_session, val_loader, "ONNX INT8"
)

Evaluating PyTorch FP32:   0%|          | 0/79 [00:00<?, ?it/s]

Evaluating ONNX FP32:   0%|          | 0/79 [00:00<?, ?it/s]

Evaluating ONNX INT8:   0%|          | 0/79 [00:00<?, ?it/s]

In [32]:
print(f"PyTorch FP32 | Accuracy: {pytorch_acc:.4f} | Macro F1: {pytorch_f1:.4f}")
print(f"ONNX FP32    | Accuracy: {onnx_acc:.4f} | Macro F1: {onnx_f1:.4f}")
print(f"ONNX INT8    | Accuracy: {int8_acc:.4f} | Macro F1: {int8_f1:.4f}")

PyTorch FP32 | Accuracy: 0.8344 | Macro F1: 0.8317
ONNX FP32    | Accuracy: 0.8344 | Macro F1: 0.8317
ONNX INT8    | Accuracy: 0.7974 | Macro F1: 0.7951


In [33]:
import time
import numpy as np

def benchmark_latency(inference_fn, model, images, warmup=50, iterations=1000):

    images = images[:1].cpu()

    # Warm-up
    for _ in range(warmup):
        inference_fn(model, images)

    latencies = []

    # Benchmark
    for _ in range(iterations):
        start = time.perf_counter()

        inference_fn(model, images)

        end = time.perf_counter()
        latencies.append((end - start) * 1000)

    return {
        "mean_ms": np.mean(latencies),
        "median_ms": np.median(latencies),
        "p95_ms": np.percentile(latencies, 95)
    }

In [34]:
images, _ = next(iter(val_loader))

pytorch_latency = benchmark_latency(
    pytorch_inference, model, images
)

onnx_latency = benchmark_latency(
    onnx_inference, fp32_session, images
)

int8_latency = benchmark_latency(
    onnx_int8_inference, int8_session, images
)

print("PyTorch FP32:", pytorch_latency)
print("ONNX FP32:", onnx_latency)
print("ONNX INT8:", int8_latency)

PyTorch FP32: {'mean_ms': np.float64(57.96564062999278), 'median_ms': np.float64(52.7461689998745), 'p95_ms': np.float64(79.46795005032072)}
ONNX FP32: {'mean_ms': np.float64(36.97726912199869), 'median_ms': np.float64(33.63981800021065), 'p95_ms': np.float64(50.258394899947234)}
ONNX INT8: {'mean_ms': np.float64(26.029468087992427), 'median_ms': np.float64(23.916453500078205), 'p95_ms': np.float64(34.682083250345386)}


In [35]:
import time
import numpy as np

def benchmark_throughput(
    inference_fn,
    model,
    images,
    batch_sizes=(1, 16, 32, 64),
    warmup=20,
    iterations=100
):
    results = {}

    for batch_size in batch_sizes:
        batch = images[:batch_size].cpu()

        # Warm-up
        for _ in range(warmup):
            inference_fn(model, batch)

        latencies = []

        # Benchmark
        for _ in range(iterations):
            start = time.perf_counter()
            inference_fn(model, batch)
            end = time.perf_counter()

            latencies.append(end - start)

        mean_latency = np.mean(latencies)

        throughput = batch_size / mean_latency

        results[batch_size] = {
            "mean_latency_ms": mean_latency * 1000,
            "throughput_img_sec": throughput
        }

    return results

In [36]:
images, _ = next(iter(val_loader))

pytorch_throughput = benchmark_throughput(
    pytorch_inference, model, images
)

onnx_throughput = benchmark_throughput(
    onnx_inference, fp32_session, images
)

int8_throughput = benchmark_throughput(
    onnx_int8_inference, int8_session, images
)

print("PyTorch FP32:", pytorch_throughput)
print("ONNX FP32:", onnx_throughput)
print("ONNX INT8:", int8_throughput)

PyTorch FP32: {1: {'mean_latency_ms': np.float64(55.59884493005484), 'throughput_img_sec': np.float64(17.985985163145614)}, 16: {'mean_latency_ms': np.float64(833.175410050053), 'throughput_img_sec': np.float64(19.203639242112057)}, 32: {'mean_latency_ms': np.float64(1797.5282359499852), 'throughput_img_sec': np.float64(17.80222383159848)}, 64: {'mean_latency_ms': np.float64(3934.912593650015), 'throughput_img_sec': np.float64(16.264656069687625)}}
ONNX FP32: {1: {'mean_latency_ms': np.float64(42.09492863000378), 'throughput_img_sec': np.float64(23.755830750767334)}, 16: {'mean_latency_ms': np.float64(569.2399196500264), 'throughput_img_sec': np.float64(28.107656275822922)}, 32: {'mean_latency_ms': np.float64(1129.892357799954), 'throughput_img_sec': np.float64(28.321281916012)}, 64: {'mean_latency_ms': np.float64(2257.731578749963), 'throughput_img_sec': np.float64(28.347036734736577)}}
ONNX INT8: {1: {'mean_latency_ms': np.float64(26.76870829002837), 'throughput_img_sec': np.float64(

In [37]:
import os

model_files = {
    "PyTorch FP32": [
        "best_resnet18_cifar100.pth"
    ],
    "ONNX FP32": [
        "resnet18_finetuned_fp32.onnx",
        "resnet18_finetuned_fp32.onnx.data"
    ],
    "ONNX INT8": [
        "resnet18_finetuned_int8.onnx"
    ]
}

model_sizes = {}

for model_name, files in model_files.items():
    total_size = 0

    print(f"\n{model_name}:")

    for file in files:
        if os.path.exists(file):
            size = os.path.getsize(file)
            total_size += size
            print(f"  {file}: {size / (1024**2):.2f} MB")

    model_sizes[model_name] = total_size / (1024**2)

    print(f"  Total: {model_sizes[model_name]:.2f} MB")


PyTorch FP32:
  best_resnet18_cifar100.pth: 42.91 MB
  Total: 42.91 MB

ONNX FP32:
  resnet18_finetuned_fp32.onnx: 0.09 MB
  resnet18_finetuned_fp32.onnx.data: 42.81 MB
  Total: 42.90 MB

ONNX INT8:
  resnet18_finetuned_int8.onnx: 10.86 MB
  Total: 10.86 MB


In [38]:
fp32_size = model_sizes["ONNX FP32"]
int8_size = model_sizes["ONNX INT8"]

size_reduction = (1 - int8_size / fp32_size) * 100
compression_ratio = fp32_size / int8_size

print(f"Size reduction: {size_reduction:.2f}%")
print(f"Compression ratio: {compression_ratio:.2f}x")

Size reduction: 74.67%
Compression ratio: 3.95x


In [39]:
test_pytorch_acc, test_pytorch_f1 = evaluate_model(
    pytorch_inference,
    model,
    test_loader,
    "PyTorch FP32 Test"
)

test_onnx_acc, test_onnx_f1 = evaluate_model(
    onnx_inference,
    fp32_session,
    test_loader,
    "ONNX FP32 Test"
)

test_int8_acc, test_int8_f1 = evaluate_model(
    onnx_int8_inference,
    int8_session,
    test_loader,
    "ONNX INT8 Test"
)

Evaluating PyTorch FP32 Test:   0%|          | 0/157 [00:00<?, ?it/s]

Evaluating ONNX FP32 Test:   0%|          | 0/157 [00:00<?, ?it/s]

Evaluating ONNX INT8 Test:   0%|          | 0/157 [00:00<?, ?it/s]

In [40]:
print(f"PyTorch FP32 | Accuracy: {test_pytorch_acc:.4f} | Macro F1: {test_pytorch_f1:.4f}")
print(f"ONNX FP32    | Accuracy: {test_onnx_acc:.4f} | Macro F1: {test_onnx_f1:.4f}")
print(f"ONNX INT8    | Accuracy: {test_int8_acc:.4f} | Macro F1: {test_int8_f1:.4f}")

PyTorch FP32 | Accuracy: 0.8272 | Macro F1: 0.8271
ONNX FP32    | Accuracy: 0.8272 | Macro F1: 0.8271
ONNX INT8    | Accuracy: 0.7998 | Macro F1: 0.7998
